In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets
import torchvision.transforms.v2 as transforms
import torchvision.models as models

In [2]:
# 1.디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
#사전 학습된 ResNet50 모델 로드
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

print(f'model : {dir(model)}')
print(f'model:{model.fc}')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\playdata/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 66.8MB/s]


model : ['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_impl', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_make_layer', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_norm_la

In [4]:
#ResNet50의 마지막 Fully Connected Layer(model.fc) 교체 (FashionMNIST 클래스 수 = 10개)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

model = model.to(device)

In [ ]:
########################################################
# 1. 학습/검증 데이터 전처리 파이프라인 분리 (v2 적용)
########################################################

# 1-1. 학습 데이터용 파이프라인 (ToImage 추가)
train_transform = transforms.Compose([
    transforms.ToImage(),                                # PIL 이미지를 Image Tensor로 변환
    transforms.Resize((224, 224)),                       # 224x224 리사이징
    transforms.Grayscale(num_output_channels=3),        # 1채널 -> 3채널(RGB) 변환
    transforms.RandomHorizontalFlip(p=0.5),             # 50% 확률로 좌우 반전
    transforms.RandomRotation(degrees=15),               # ±15도 범위 내 랜덤 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2),# Brightness/Contrast 변형
    transforms.ToDtype(torch.float32, scale=True),      # float32 변환 및 0~1 스케일링
    transforms.Normalize(                               # ImageNet 사전 학습 모델 표준 정규화
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# 1-2. 검증 및 테스트 데이터용 파이프라인 (ToImage 추가)
val_test_transform = transforms.Compose([
    transforms.ToImage(),                                # PIL 이미지를 Image Tensor로 변환
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

########################################################
# 2. 데이터셋 분할 및 Transform 각각 적용
########################################################

# Subset 개별 transform 적용을 위한 커스텀 Dataset 클래스
class TransformedDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# 전체 데이터 로드 (원본 상태)
full_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=val_test_transform)

# Train / Validation 분할 (55,000 / 5,000)
train_size = 55000
val_size = 5000
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# 각 Split에 맞는 Transform 지정
train_dataset = TransformedDataset(train_subset, transform=train_transform)
val_dataset = TransformedDataset(val_subset, transform=val_test_transform)

# 데이터로더 구축
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

########################################################
# 3. ResNet50 모델 설정 및 FC Layer 변경
########################################################
# 사전 학습된 ResNet50 모델 로드
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# ResNet50의 마지막 Fully Connected Layer(model.fc) 교체 (FashionMNIST 클래스 수 = 10개)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

model = model.to(device)

########################################################
# 4. 손실 함수, 옵티마이저 설정
########################################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

########################################################
# 5. 학습 및 검증 루프 (Training & Validation Loop)
########################################################
epochs = 5

for epoch in range(epochs):
    # --- [Training Phase (증강 적용)] ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    train_epoch_loss = train_loss / train_total
    train_epoch_acc = train_correct / train_total

    # --- [Validation Phase (증강 미적용)] ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total
    
    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_epoch_loss:.4f}, Train Acc: {train_epoch_acc*100:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc*100:.2f}%")

########################################################
# 6. 최종 테스트 평가 (Test Phase)
########################################################
model.eval()
test_loss, test_correct, test_total = 0.0, 0, 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_epoch_loss = test_loss / test_total
test_epoch_acc = test_correct / test_total
print(f"\n[Test Result] Loss: {test_epoch_loss:.4f}, Accuracy: {test_epoch_acc*100:.2f}%")